# LeetCode 2089 — Find Target Indices After Sorting Array bằng Python

Link đề: https://leetcode.com/problems/find-target-indices-after-sorting-array/ — Độ khó: **Easy**

Skeleton LeetCode:

```python
class Solution:
    def targetIndices(self, nums: List[int], target: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **sắp xếp trước**, rồi mới hỏi "`target` nằm ở những chỉ số nào?".
2. Biết dùng `sorted`, `enumerate`, list comprehension và `bisect` trong Python.
3. Viết được 4 cách: sort rồi quét, đếm (không cần sort thật), binary search, counting sort.
4. Nhận ra insight: các `target` sau khi sort nằm **liền nhau**, nên chỉ cần biết *có bao nhiêu số nhỏ hơn target* và *có bao nhiêu số bằng target*.
5. Tự trace, tự viết test ngẫu nhiên đối chiếu các cách, và tránh các lỗi hay gặp (quên sort, trả về chỉ số gốc...).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (đánh số từ 0) và một số `target`. **Sắp xếp `nums` theo thứ tự không giảm**, rồi trả về danh sách **tăng dần** các chỉ số `i` sao cho `nums[i] == target` (trên mảng đã sắp xếp). Nếu không có thì trả về `[]`.

Ví dụ của đề:

| nums | target | sau khi sort | kết quả |
|---|---|---|---|
| `[1,2,5,2,3]` | 2 | `[1,2,2,3,5]` | `[1,2]` |
| `[1,2,5,2,3]` | 3 | `[1,2,2,3,5]` | `[3]` |
| `[1,2,5,2,3]` | 5 | `[1,2,2,3,5]` | `[4]` |
| `[1,2,5,2,3]` | 4 | `[1,2,2,3,5]` | `[]` |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i], target <= 100`.

Chú ý: chỉ số trả về là chỉ số **trong mảng đã sort**, không phải chỉ số trong mảng gốc.

In [1]:
examples = [
    ([1, 2, 5, 2, 3], 2, [1, 2]),
    ([1, 2, 5, 2, 3], 3, [3]),
    ([1, 2, 5, 2, 3], 5, [4]),
    ([1, 2, 5, 2, 3], 4, []),
]
for nums, target, expected in examples:
    print("nums =", nums, "| target =", target, "| sorted =", sorted(nums), "| mong đợi =", expected)

nums = [1, 2, 5, 2, 3] | target = 2 | sorted = [1, 2, 2, 3, 5] | mong đợi = [1, 2]
nums = [1, 2, 5, 2, 3] | target = 3 | sorted = [1, 2, 2, 3, 5] | mong đợi = [3]
nums = [1, 2, 5, 2, 3] | target = 5 | sorted = [1, 2, 2, 3, 5] | mong đợi = [4]
nums = [1, 2, 5, 2, 3] | target = 4 | sorted = [1, 2, 2, 3, 5] | mong đợi = []


## 3. Kiến thức nền

### 3.1. `sorted(a)` và `a.sort()`

- `sorted(a)` trả về **list mới** đã sắp xếp, `a` giữ nguyên.
- `a.sort()` sắp xếp **tại chỗ** và trả về `None` (lỗi kinh điển: `b = a.sort()` làm `b` thành `None`).

In [2]:
a = [5, 1, 4, 1]
b = sorted(a)
print("a =", a, "| b =", b)
c = a.sort()
print("a sau sort() =", a, "| giá trị trả về =", c)

a = [5, 1, 4, 1] | b = [1, 1, 4, 5]
a sau sort() = [1, 1, 4, 5] | giá trị trả về = None


### 3.2. `enumerate` và list comprehension

`enumerate(a)` sinh ra cặp `(chỉ số, giá trị)`. List comprehension `[i for i, x in enumerate(a) if x == t]` gom mọi chỉ số `i` mà `a[i] == t` — đúng thứ ta cần sau khi sort.

In [3]:
a = [1, 2, 2, 3, 5]
for i, x in enumerate(a):
    print("i =", i, "x =", x)
print([i for i, x in enumerate(a) if x == 2])

i = 0 x = 1
i = 1 x = 2
i = 2 x = 2
i = 3 x = 3
i = 4 x = 5
[1, 2]


### 3.3. `list.count(x)` và đếm số phần tử nhỏ hơn

`a.count(x)` đếm số lần `x` xuất hiện. Số phần tử nhỏ hơn `t` có thể đếm bằng `sum(1 for x in a if x < t)`, vì `True` được cộng như `1`.

In [4]:
a = [1, 2, 5, 2, 3]
print("count(2) =", a.count(2))
print("số phần tử < 3 =", sum(1 for x in a if x < 3))
print("tổng bool:", sum([True, False, True]))

count(2) = 2
số phần tử < 3 = 3
tổng bool: 2


### 3.4. `range(start, stop)` và module `bisect`

`range(l, l + c)` sinh `c` số nguyên liên tiếp bắt đầu từ `l`. Với list đã sort, `bisect_left(a, t)` trả về vị trí đầu tiên có thể chèn `t` mà vẫn sorted (= số phần tử `< t`), còn `bisect_right(a, t)` trả về vị trí **sau** mọi số bằng `t` (= số phần tử `<= t`).

In [5]:
from bisect import bisect_left, bisect_right
a = [1, 2, 2, 3, 5]
print(list(range(3, 3 + 2)))
print("bisect_left(2) =", bisect_left(a, 2), "| bisect_right(2) =", bisect_right(a, 2))
print("bisect_left(4) =", bisect_left(a, 4), "| bisect_right(4) =", bisect_right(a, 4))

[3, 4]
bisect_left(2) = 1 | bisect_right(2) = 3
bisect_left(4) = 4 | bisect_right(4) = 4


## 4. Ý tưởng nền tảng — tính bằng tay

`nums = [1,2,5,2,3]`, `target = 2`. Sort ra `[1,2,2,3,5]`, chỉ số: 0,1,2,3,4. Các số `2` nằm ở chỉ số 1 và 2.

Nhận xét: trước chúng có đúng **1** số nhỏ hơn 2 (số `1`), nên số `2` đầu tiên có chỉ số 1. Có **2** số bằng 2 nên chúng chiếm chỉ số `1, 2`. Ta chưa cần sort thật vẫn đoán được kết quả!

In [6]:
nums, target = [1, 2, 5, 2, 3], 2
less = sum(1 for x in nums if x < target)
equal = nums.count(target)
print("less =", less, "| equal =", equal, "| kết quả =", list(range(less, less + equal)))

less = 1 | equal = 2 | kết quả = [1, 2]


## 5. Cách 1 — Brute Force: sort rồi quét

Ý tưởng đúng theo đề: sort mảng, duyệt từng chỉ số, chỉ số nào chứa `target` thì thêm vào kết quả.

In [7]:
def indices_sort_scan(nums, target):
    a = sorted(nums)
    res = []
    for i in range(len(a)):
        if a[i] == target:
            res.append(i)
    return res

for nums, target, expected in examples:
    print(indices_sort_scan(nums, target), expected)

[1, 2] [1, 2]
[3] [3]
[4] [4]
[] []


Độ phức tạp: sort `O(n log n)`, quét `O(n)` → **Time `O(n log n)`**, Space `O(n)` cho bản sao.

Nút thắt: ta sort chỉ để biết "những số bằng target nằm ở đâu", nhưng thứ tự giữa các số **khác** target không quan trọng. Mất công sắp xếp toàn bộ là thừa.

## 6. Ý tưởng thuật toán / bước đột phá

Sau khi sort, mọi số bằng `target` đứng **liền nhau** thành một khối. Khối này bắt đầu tại chỉ số `less` (số phần tử nhỏ hơn target) và dài `equal` phần tử.

- Nếu `equal == 0` → khối rỗng → kết quả `[]` (`range(less, less)` rỗng sẵn).
- Nếu `equal > 0` → kết quả `[less, less+1, ..., less+equal-1]`.

Invariant: vị trí của một giá trị `v` trong mảng đã sort chỉ phụ thuộc vào *có bao nhiêu phần tử nhỏ hơn `v`*, không phụ thuộc thứ tự giữa các phần tử còn lại.

## 7. Các cách tiếp theo

### Cách 2 — Đếm `less` và `equal` (một lượt quét, không sort)

Duyệt `nums` một lần: gặp `x < target` thì `less += 1`; gặp `x == target` thì `equal += 1`. Trả về `list(range(less, less + equal))`.

In [8]:
def indices_count(nums, target):
    less = equal = 0
    for x in nums:
        if x < target:
            less += 1
        elif x == target:
            equal += 1
    return list(range(less, less + equal))

for nums, target, expected in examples:
    print(indices_count(nums, target), expected)

[1, 2] [1, 2]
[3] [3]
[4] [4]
[] []


In [9]:
from typing import List

class SolutionCount:
    def targetIndices(self, nums: List[int], target: int) -> List[int]:
        less = equal = 0
        for x in nums:
            if x < target:
                less += 1
            elif x == target:
                equal += 1
        return list(range(less, less + equal))

print(SolutionCount().targetIndices([1, 2, 5, 2, 3], 2))

[1, 2]


Độ phức tạp: **Time `O(n)`**, **Space `O(1)`** (không tính list kết quả).

### Cách 3 — Sort + binary search (`bisect`)

Sau khi sort, `lo = bisect_left(a, target)` là chỉ số đầu khối, `hi = bisect_right(a, target)` là chỉ số *sau* khối. Kết quả là `list(range(lo, hi))`. Nếu không có target thì `lo == hi`.

In [10]:
from bisect import bisect_left, bisect_right

def indices_bisect(nums, target):
    a = sorted(nums)
    lo = bisect_left(a, target)
    hi = bisect_right(a, target)
    return list(range(lo, hi))

for nums, target, expected in examples:
    print(indices_bisect(nums, target), expected)

[1, 2] [1, 2]
[3] [3]
[4] [4]
[] []


In [11]:
from typing import List
from bisect import bisect_left, bisect_right

class SolutionBisect:
    def targetIndices(self, nums: List[int], target: int) -> List[int]:
        a = sorted(nums)
        return list(range(bisect_left(a, target), bisect_right(a, target)))

print(SolutionBisect().targetIndices([1, 2, 5, 2, 3], 5))

[4]


Độ phức tạp: sort `O(n log n)` chiếm ưu thế, hai lần bisect `O(log n)` → **Time `O(n log n)`**, Space `O(n)`. Cách này hay khi mảng *đã sort sẵn* và hỏi nhiều `target`: mỗi truy vấn chỉ `O(log n)`.

### Cách 4 — Counting sort (vì `nums[i] <= 100`)

Giá trị chỉ từ 1 đến 100, nên đếm số lần xuất hiện của từng giá trị vào mảng `cnt[0..100]`. Số phần tử nhỏ hơn `target` là `sum(cnt[:target])`, số phần tử bằng target là `cnt[target]`.

In [12]:
def indices_counting(nums, target):
    cnt = [0] * 101
    for x in nums:
        cnt[x] += 1
    start = sum(cnt[:target])
    return list(range(start, start + cnt[target]))

for nums, target, expected in examples:
    print(indices_counting(nums, target), expected)

[1, 2] [1, 2]
[3] [3]
[4] [4]
[] []


In [13]:
from typing import List

class SolutionCountingSort:
    def targetIndices(self, nums: List[int], target: int) -> List[int]:
        cnt = [0] * 101
        for x in nums:
            cnt[x] += 1
        start = sum(cnt[:target])
        return list(range(start, start + cnt[target]))

print(SolutionCountingSort().targetIndices([1, 2, 5, 2, 3], 3))

[3]


Độ phức tạp: **Time `O(n + V)`** với `V = 100`, **Space `O(V)`**. Khi `V` cố định nhỏ, đây là `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 với `nums = [1,2,5,2,3]`, `target = 2`:

| x | so với 2 | less | equal |
|---|---|---|---|
| 1 | nhỏ hơn | 1 | 0 |
| 2 | bằng | 1 | 1 |
| 5 | lớn hơn | 1 | 1 |
| 2 | bằng | 1 | 2 |
| 3 | lớn hơn | 1 | 2 |

Kết quả `range(1, 3) = [1, 2]`.

In [14]:
def trace_count(nums, target):
    less = equal = 0
    for step, x in enumerate(nums, 1):
        if x < target:
            less += 1
            note = "nhỏ hơn"
        elif x == target:
            equal += 1
            note = "bằng"
        else:
            note = "lớn hơn"
        print(f"bước {step}: x={x} ({note}) -> less={less}, equal={equal}")
    print("kết quả = range(%d, %d) =" % (less, less + equal), list(range(less, less + equal)))

trace_count([1, 2, 5, 2, 3], 2)

bước 1: x=1 (nhỏ hơn) -> less=1, equal=0
bước 2: x=2 (bằng) -> less=1, equal=1
bước 3: x=5 (lớn hơn) -> less=1, equal=1
bước 4: x=2 (bằng) -> less=1, equal=2
bước 5: x=3 (lớn hơn) -> less=1, equal=2
kết quả = range(1, 3) = [1, 2]


In [15]:
def trace_sort_scan(nums, target):
    a = sorted(nums)
    print("sau sort:", a)
    res = []
    for i, x in enumerate(a):
        hit = (x == target)
        if hit:
            res.append(i)
        print(f"i={i}, a[i]={x}, bằng target? {hit}, res={res}")
    return res

trace_sort_scan([3, 3, 1, 4, 3], 3)

sau sort: [1, 3, 3, 3, 4]
i=0, a[i]=1, bằng target? False, res=[]
i=1, a[i]=3, bằng target? True, res=[1]
i=2, a[i]=3, bằng target? True, res=[1, 2]
i=3, a[i]=3, bằng target? True, res=[1, 2, 3]
i=4, a[i]=4, bằng target? False, res=[1, 2, 3]


[1, 2, 3]

## 9. Code nộp LeetCode

Cách `O(n)` đếm là gọn nhất, nộp như sau:

In [16]:
from typing import List

class Solution:
    def targetIndices(self, nums: List[int], target: int) -> List[int]:
        less = equal = 0
        for x in nums:
            if x < target:
                less += 1
            elif x == target:
                equal += 1
        return list(range(less, less + equal))

print(Solution().targetIndices([1, 2, 5, 2, 3], 2))

[1, 2]


## 10. Vì sao thuật toán đúng?

Xét mảng đã sort `a`. Mọi phần tử `< target` phải đứng trước mọi phần tử `>= target`, nên có đúng `less` phần tử đứng ở các chỉ số `0..less-1`. Tiếp theo là toàn bộ `equal` phần tử bằng `target` (chúng giống nhau, nên chiếm đúng `equal` ô liên tiếp) tại chỉ số `less..less+equal-1`. Phần còn lại `> target` đứng sau. Vậy tập chỉ số chứa `target` chính xác là `[less, less+equal)`, và `range` cho ra thứ tự tăng dần như đề yêu cầu. Khi `equal = 0` thì khoảng rỗng, tương ứng `[]`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 100` là giá trị lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Sort + quét | `O(n log n)` | `O(n)` |
| 2. Đếm | `O(n)` | `O(1)` |
| 3. Sort + bisect | `O(n log n)` | `O(n)` |
| 4. Counting sort | `O(n + V)` | `O(V)` |

Trường hợp xấu nhất và trung bình giống nhau ở cả bốn cách. Kích thước kết quả có thể tới `n` (khi mọi phần tử bằng target), nên "Space `O(1)`" của Cách 2 không tính list trả về.

## 12. Bộ test

Ta kiểm các ví dụ đề, edge case (một phần tử, tất cả bằng nhau, target không có, target nhỏ nhất/lớn nhất), rồi đối chiếu cả 4 cách với input ngẫu nhiên có `random.seed`.

In [17]:
cases = [
    ([1, 2, 5, 2, 3], 2, [1, 2]),
    ([1, 2, 5, 2, 3], 3, [3]),
    ([1, 2, 5, 2, 3], 5, [4]),
    ([1, 2, 5, 2, 3], 4, []),
    ([7], 7, [0]),
    ([7], 3, []),
    ([4, 4, 4, 4], 4, [0, 1, 2, 3]),
    ([100, 1, 100, 1], 1, [0, 1]),
    ([100, 1, 100, 1], 100, [2, 3]),
    ([5, 1, 3], 2, []),
]
impls = [indices_sort_scan, indices_count, indices_bisect, indices_counting,
         Solution().targetIndices, SolutionCount().targetIndices,
         SolutionBisect().targetIndices, SolutionCountingSort().targetIndices]
for nums, target, expected in cases:
    for f in impls:
        assert f(list(nums), target) == expected, (f.__name__, nums, target)
print("Tất cả", len(cases), "ca x", len(impls), "cách đều đúng")

Tất cả 10 ca x 8 cách đều đúng


In [18]:
import random
random.seed(2089)
for _ in range(3000):
    n = random.randint(1, 30)
    nums = [random.randint(1, 8) for _ in range(n)]
    target = random.randint(1, 9)
    want = indices_sort_scan(nums, target)
    assert indices_count(nums, target) == want
    assert indices_bisect(nums, target) == want
    assert indices_counting(nums, target) == want
    assert Solution().targetIndices(nums, target) == want
print("3000 input ngẫu nhiên: 4 cách khớp nhau")

3000 input ngẫu nhiên: 4 cách khớp nhau


In [19]:
# phản ví dụ: quên sort sẽ sai ngay ví dụ 1
def wrong_no_sort(nums, target):
    return [i for i, x in enumerate(nums) if x == target]

print("không sort:", wrong_no_sort([1, 2, 5, 2, 3], 2), "| đúng:", [1, 2])

không sort: [1, 3] | đúng: [1, 2]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên sắp xếp, trả về chỉ số gốc

Đề hỏi chỉ số sau khi sort. Dùng `enumerate(nums)` trực tiếp cho chỉ số gốc `[1, 3]` thay vì `[1, 2]`.

In [20]:
def wrong1(nums, target):
    return [i for i, x in enumerate(nums) if x == target]   # SAI: chưa sort

def right1(nums, target):
    return [i for i, x in enumerate(sorted(nums)) if x == target]

print(wrong1([1, 2, 5, 2, 3], 2), right1([1, 2, 5, 2, 3], 2))

[1, 3] [1, 2]


### Lỗi 2: `nums = nums.sort()`

`sort()` trả về `None`, nên `nums` thành `None` và vòng lặp phía sau báo `TypeError`.

In [21]:
nums = [3, 1, 2]
nums = nums.sort()
print(nums)  # None
nums = [3, 1, 2]
nums.sort()      # đúng: sort tại chỗ, không gán lại
print(nums)

None
[1, 2, 3]


### Lỗi 3: Dùng `bisect_right - 1` hoặc nhầm hai hàm bisect

`bisect_right` trả về vị trí *sau* khối target. Quên `-1` hoặc dùng `bisect_left` cho cả hai đầu sẽ cho kết quả rỗng hoặc lệch một.

In [22]:
a = sorted([1, 2, 5, 2, 3])
print("sai (left, left):", list(range(bisect_left(a, 2), bisect_left(a, 2))))
print("sai (lệch hi):   ", list(range(bisect_left(a, 2), bisect_right(a, 2) + 1)))
print("đúng:            ", list(range(bisect_left(a, 2), bisect_right(a, 2))))

sai (left, left): []
sai (lệch hi):    [1, 2, 3]
đúng:             [1, 2]


### Lỗi 4: Đếm `<=` thay vì `<` cho `less`

Nếu `less` đếm cả số bằng target thì khối bị đẩy lùi. Với `[1,2,5,2,3]`, target 2: `less` sai = 3 thay vì 1.

In [23]:
nums, target = [1, 2, 5, 2, 3], 2
less_wrong = sum(1 for x in nums if x <= target)
less_right = sum(1 for x in nums if x < target)
print(less_wrong, less_right)

3 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có chữ "sau khi sắp xếp" nhưng chỉ hỏi vị trí của **một giá trị**: nghĩ ngay tới *đếm số phần tử nhỏ hơn* thay vì sort.
- Giá trị nhỏ (≤ 100) gợi ý **counting sort** / mảng đếm.
- Cần vị trí đầu/cuối của giá trị trong mảng sort: `bisect_left` / `bisect_right`.
- Công thức "vị trí = số phần tử nhỏ hơn" cũng là cốt lõi của **rank**, **counting sort ổn định** và bài "How Many Numbers Are Smaller Than the Current Number".
- Kiểm tra nhanh: `len(kết quả) == nums.count(target)`.

## 15. Biến thể của bài

1. Trả về chỉ số **trong mảng gốc** của các phần tử bằng target (không sort): chỉ cần quét một lần.
2. Chỉ trả về **chỉ số đầu tiên** hoặc **cuối cùng** của target sau khi sort (`bisect_left` / `bisect_right - 1`).
3. Giá trị lớn tới `10^9`: counting sort không còn dùng được, dùng Cách 2 vẫn `O(n)`.
4. Sort **giảm dần**: `less` đổi thành số phần tử *lớn hơn* target.
5. Nhiều truy vấn `target` trên cùng mảng: sort một lần rồi bisect cho mỗi truy vấn.

In [24]:
def first_index_after_sort(nums, target):
    less = sum(1 for x in nums if x < target)
    return less if target in nums else -1

def indices_descending(nums, target):
    greater = sum(1 for x in nums if x > target)
    return list(range(greater, greater + nums.count(target)))

print(first_index_after_sort([1, 2, 5, 2, 3], 2), first_index_after_sort([1, 2, 5, 2, 3], 4))
print(indices_descending([1, 2, 5, 2, 3], 2), "| sorted giảm:", sorted([1, 2, 5, 2, 3], reverse=True))

1 -1
[2, 3] | sorted giảm: [5, 3, 2, 2, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sort + quét | làm đúng nghĩa đen đề | `O(n log n)` | `O(n)` | dễ viết, người mới |
| 2. Đếm less/equal | vị trí = số phần tử nhỏ hơn | `O(n)` | `O(1)` | nộp bài, tốt nhất |
| 3. Sort + bisect | tìm biên khối bằng binary search | `O(n log n)` | `O(n)` | mảng đã sort / nhiều truy vấn |
| 4. Counting sort | mảng đếm theo giá trị | `O(n+V)` | `O(V)` | giá trị nhỏ, biết trước |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `nums = [4, 1, 4, 2]`, `target = 4`. In ra gì? Hãy tự dự đoán trước khi chạy.

**Bài 2 (tính tay):** `nums = [9, 3, 3, 3, 1]`, `target = 3`. Tính `less`, `equal` và kết quả.

**Bài 3 (điền TODO):** hoàn thành `exercise_count` bằng ý tưởng đếm.

**Bài 4 (tự viết lại):** viết `exercise_bisect(nums, target)` dùng `bisect`.

**Bài 5 (mini challenge):** viết `exercise_last(nums, target)` trả về chỉ số **cuối cùng** của target sau khi sort, hoặc `-1` nếu không có.

In [25]:
# Bài 1: chạy để so với dự đoán
print(indices_count([4, 1, 4, 2], 4))

[2, 3]


In [26]:
def exercise_count(nums, target):
    less = 0
    equal = 0
    # TODO: duyệt nums, tăng less khi x < target, tăng equal khi x == target
    # TODO: return list(range(less, less + equal))
    pass

# (không gọi khi chưa làm xong)

In [27]:
def exercise_bisect(nums, target):
    # TODO: a = sorted(nums); dùng bisect_left và bisect_right
    pass

def exercise_last(nums, target):
    # TODO: tìm chỉ số cuối của target sau khi sort, -1 nếu không có
    pass

# (không gọi khi chưa làm xong)

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Sort trước: `[1,2,4,4]` | Hai số 4 nằm cạnh nhau | Chỉ số 2 và 3 |
| 2 | Sort: `[1,3,3,3,9]` | `less` = số phần tử < 3 | `less=1`, `equal=3` → `[1,2,3]` |
| 3 | Dùng `if/elif` trong vòng `for` | `x < target` → `less`, `x == target` → `equal` | Trả `list(range(less, less+equal))` |
| 4 | `a = sorted(nums)` | Hai biên: `bisect_left`, `bisect_right` | `list(range(lo, hi))` |
| 5 | Cuối khối là `bisect_right - 1` | Chỉ khi `target` có mặt | `less + equal - 1`, hoặc `-1` nếu `equal == 0` |

## 19. Đáp án tham khảo cho bài tập

Bài 1: sort `[1,2,4,4]` → `[2, 3]`. Bài 2: `less=1`, `equal=3` → `[1, 2, 3]`.

In [28]:
def exercise_count(nums, target):
    less = equal = 0
    for x in nums:
        if x < target:
            less += 1
        elif x == target:
            equal += 1
    return list(range(less, less + equal))

def exercise_bisect(nums, target):
    a = sorted(nums)
    return list(range(bisect_left(a, target), bisect_right(a, target)))

def exercise_last(nums, target):
    a = sorted(nums)
    hi = bisect_right(a, target)
    return hi - 1 if hi > bisect_left(a, target) else -1

assert indices_count([4, 1, 4, 2], 4) == [2, 3]
assert exercise_count([9, 3, 3, 3, 1], 3) == [1, 2, 3]
assert exercise_bisect([9, 3, 3, 3, 1], 3) == [1, 2, 3]
assert exercise_bisect([1, 2], 5) == []
assert exercise_last([1, 2, 5, 2, 3], 2) == 2
assert exercise_last([1, 2, 5, 2, 3], 4) == -1
print("Đáp án bài tập đều đúng")

Đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (đếm):**

```python
less = equal = 0
for x in nums:
    if x < target: less += 1
    elif x == target: equal += 1
return list(range(less, less + equal))
```

**Checklist:**

- [ ] Chỉ số trả về là của mảng **đã sort**, tăng dần?
- [ ] `less` đếm `<` (không phải `<=`)?
- [ ] Target không tồn tại → `[]` (không `-1`)?
- [ ] Không gán `nums = nums.sort()`?
- [ ] Đã thử: một phần tử, tất cả bằng nhau, target nhỏ nhất/lớn nhất?